# 🦅 Horus-OSINT: Fine-Tuning Llama-3-8B-Instruct
**CISC 886 – Cloud Computing | Queen's University**

**Student NetID:** 25BBDF

This notebook fine-tunes `Meta-Llama-3-8B-Instruct` on the processed GTD+GDELT OSINT dataset
using **Unsloth** (2x faster training) and **QLoRA** (4-bit, PEFT) as recommended by the course resource guide.

### Pipeline:
1. Install Unsloth + dependencies
2. Load base model in 4-bit quantization
3. Apply LoRA adapters (PEFT)
4. Load processed JSONL data from S3
5. Train with SFTTrainer
6. Compare base vs fine-tuned responses
7. Export to GGUF → upload to S3 for EC2 deployment

---
> ⚠️ **Runtime:** Set to `T4 GPU` in Google Colab (Runtime → Change runtime type → T4 GPU)

## Section 1 — Install Dependencies

In [ ]:
# Install Unsloth — optimized for Colab T4 (CUDA 12.1, PyTorch 2.3)
# Unsloth provides 2x faster training and 60% less VRAM than standard HuggingFace
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes -q

# AWS SDK to pull processed data from S3
!pip install boto3 -q

print("✅ All dependencies installed.")

In [ ]:
# Verify GPU is available — required for training
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU detected: {gpu_name}")
print(f"   VRAM available: {vram_gb:.1f} GB")
print(f"   CUDA version:   {torch.version.cuda}")

## Section 2 — Configuration
All project-wide constants are defined here. Edit only this cell to adapt the notebook.

In [ ]:
# ─────────────────────────────────────────────
#  PROJECT CONFIGURATION  —  edit here only
# ─────────────────────────────────────────────

NETID          = "25BBDF"                          # Queen's NetID
S3_BUCKET      = f"{NETID}-bucket"                 # e.g. 25BBDF-bucket
S3_DATA_KEY    = "processed/train.jsonl"           # path to processed JSONL in S3
S3_MODEL_KEY   = "models/horus-llama3-osint.gguf" # where GGUF will be uploaded
AWS_REGION     = "us-east-1"

# Model — Meta-Llama-3-8B-Instruct from Unsloth's optimized hub
BASE_MODEL     = "unsloth/Meta-Llama-3-8B-Instruct-bnb-4bit"
MODEL_NAME     = "horus-llama3-osint"              # local save name & Ollama model name

# ─────────────────────────────────────────────
#  HYPERPARAMETERS  (documented in report)
# ─────────────────────────────────────────────
LORA_R         = 16      # LoRA rank — balances VRAM vs expressive power
LORA_ALPHA     = 32      # Scaling factor = 2 × rank (standard practice)
LORA_DROPOUT   = 0.05    # Small dropout for regularisation
LEARNING_RATE  = 2e-4    # Stable starting point for PEFT with AdamW
BATCH_SIZE     = 2       # Optimised for T4 16GB VRAM
GRAD_ACCUM     = 4       # Effective batch = 2×4 = 8 for stable gradients
MAX_STEPS      = 500     # Sufficient to adapt formatting to OSINT domain
MAX_SEQ_LEN    = 2048    # Max token length per sample
OPTIMIZER      = "adamw_8bit"  # Memory-efficient optimizer from Unsloth
WARMUP_STEPS   = 50      # Gradual LR warm-up to prevent instability
LR_SCHEDULER   = "cosine"     # Smooth LR decay over training
SEED           = 42

print("✅ Configuration loaded.")
print(f"   Model:   {BASE_MODEL}")
print(f"   S3 src:  s3://{S3_BUCKET}/{S3_DATA_KEY}")
print(f"   S3 dst:  s3://{S3_BUCKET}/{S3_MODEL_KEY}")

## Section 3 — Load Base Model with 4-bit Quantization
We use `FastLanguageModel` from Unsloth which wraps HuggingFace and applies
optimised CUDA kernels. Loading in 4-bit (QLoRA) cuts VRAM from ~16 GB to ~6 GB.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load pre-quantized Llama-3-8B-Instruct in 4-bit from Unsloth's hub
# load_in_4bit=True activates QLoRA — model weights stored as NF4 integers
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = BASE_MODEL,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,          # Auto-detect: bf16 on Ampere+, fp16 on T4
    load_in_4bit  = True,          # QLoRA: NF4 quantization, 75% VRAM reduction
)

print(f"✅ Base model loaded: {BASE_MODEL}")
print(f"   Parameters: ~8 Billion")
print(f"   Quantization: 4-bit NF4 (QLoRA)")
print(f"   Dtype: {model.dtype}")

## Section 4 — Apply LoRA Adapters (PEFT)
Instead of updating all 8B parameters, LoRA injects small trainable matrices
into the attention layers. Only ~0.1% of parameters are trained — this is why
fine-tuning fits on a 16GB T4.

In [ ]:
# Apply LoRA adapters to the attention projection layers
# target_modules lists the specific linear layers that get adapter matrices
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    target_modules      = [
        "q_proj",   # Query projection
        "k_proj",   # Key projection
        "v_proj",   # Value projection
        "o_proj",   # Output projection
        "gate_proj",# FFN gate
        "up_proj",  # FFN up
        "down_proj",# FFN down
    ],
    bias                = "none",   # No bias terms in adapters (saves memory)
    use_gradient_checkpointing = "unsloth",  # Unsloth's optimised checkpointing
    random_state        = SEED,
    use_rslora          = False,    # Standard LoRA scaling
    loftq_config        = None,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA adapters applied.")
print(f"   Trainable parameters: {trainable:,} ({100*trainable/total:.2f}% of total)")
print(f"   Frozen parameters:    {total - trainable:,}")
print(f"   LoRA rank (r):        {LORA_R}")
print(f"   LoRA alpha:           {LORA_ALPHA}")

## Section 5 — Define Llama-3 Chat Template
Llama-3-Instruct uses a specific format with special tokens. Our PySpark job
already produced data in this format — we must use the identical template here.

In [ ]:
# Llama-3 Instruct format — must match exactly what PySpark produced
# The EOS token tells the model when to stop generating
LLAMA3_TEMPLATE = """\
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system}<|eot_id|><|start_header_id|>user<|end_header_id|>
{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{response}<|eot_id|>"""

SYSTEM_PROMPT = (
    "You are Horus, an elite Intelligence Analyst specializing in Open-Source "
    "Intelligence (OSINT) and global threat analysis. You process and analyze "
    "data from the Global Terrorism Database (GTD) and GDELT to produce "
    "structured, factual intelligence reports."
)

EOS_TOKEN = tokenizer.eos_token  # <|eot_id|> for Llama-3

def format_sample(sample):
    """
    Formats a raw JSONL sample into the Llama-3 instruct template.
    Each sample must have 'instruction' and 'output' keys.
    The 'input' field is optional context.
    """
    system      = sample.get("system", SYSTEM_PROMPT)
    instruction = sample["instruction"]
    # Combine optional input context with instruction if present
    if sample.get("input", "").strip():
        instruction = f"{instruction}\n\nContext: {sample['input']}"
    response    = sample["output"]

    text = LLAMA3_TEMPLATE.format(
        system      = system,
        instruction = instruction,
        response    = response,
    ) + EOS_TOKEN  # Append EOS so model learns to stop

    return {"text": text}

# Quick sanity check — show one formatted example
test_sample = {
    "instruction": "Provide a brief intelligence summary of the bombing incident in Afghanistan in 2019.",
    "input": "",
    "output": "According to GTD records, a Bombing/Explosion took place in Afghanistan in 2019. The incident primarily targeted civilians. The perpetrators utilized explosives, resulting in approximately 12 reported fatalities and 30 wounded."
}
print("=== Sample formatted prompt ===")
print(format_sample(test_sample)["text"][:600], "...")

## Section 6 — Load Dataset from S3
The processed JSONL file was produced by the PySpark job on EMR and saved to S3.
We download it here and load it with HuggingFace `datasets`.

In [ ]:
import boto3
import os
from datasets import load_dataset
from google.colab import userdata  # Colab Secrets for AWS credentials

# ── AWS credentials from Colab Secrets (Secrets tab in left sidebar)
# Add AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY to Colab secrets
os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = AWS_REGION

# ── Download processed JSONL from S3
LOCAL_DATA_PATH = "/content/train.jsonl"

print(f"⬇️  Downloading s3://{S3_BUCKET}/{S3_DATA_KEY} ...")
s3 = boto3.client("s3", region_name=AWS_REGION)
s3.download_file(S3_BUCKET, S3_DATA_KEY, LOCAL_DATA_PATH)
print(f"✅ Downloaded to {LOCAL_DATA_PATH}")

# ── Count lines (records) in the file
with open(LOCAL_DATA_PATH) as f:
    total_lines = sum(1 for _ in f)
print(f"   Total training samples: {total_lines:,}")

In [ ]:
from datasets import load_dataset

# Load the JSONL file into a HuggingFace Dataset object
raw_dataset = load_dataset("json", data_files=LOCAL_DATA_PATH, split="train")
print(f"✅ Dataset loaded: {len(raw_dataset):,} samples")
print(f"   Columns: {raw_dataset.column_names}")

# Apply the Llama-3 formatting template to every sample
# num_proc=4 uses all Colab CPU cores for speed
dataset = raw_dataset.map(
    format_sample,
    num_proc=4,
    desc="Formatting to Llama-3 template",
)

print(f"✅ Formatting complete.")
print("\n=== First formatted sample (preview) ===")
print(dataset[0]["text"][:500], "...")

## Section 7 — Capture Base Model Response (Before Fine-Tuning)
We record the base model's response to an OSINT query **before** training.
This will be compared side-by-side with the fine-tuned response in Section 10.

In [ ]:
# Enable native 2x faster inference mode before sampling
FastLanguageModel.for_inference(model)

TEST_PROMPT = (
    "Provide a brief intelligence summary of the Assassination incident "
    "that occurred in Dominican Republic during 1970."
)

def generate_response(prompt, max_new_tokens=256):
    """Run inference and return the model's text response."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": prompt},
    ]
    # Apply the chat template — returns tokenized input IDs
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens  = max_new_tokens,
            do_sample       = False,   # Greedy decode for determinism
            temperature     = 1.0,
            pad_token_id    = tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (slice off input)
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

base_response = generate_response(TEST_PROMPT)
print("=" * 60)
print("📋 BASE MODEL RESPONSE (before fine-tuning):")
print("=" * 60)
print(base_response)
print("=" * 60)

## Section 8 — Training with SFTTrainer
We use HuggingFace's `SFTTrainer` (Supervised Fine-Tuning) with the hyperparameters
documented in the project report. Unsloth's training loop is ~2x faster than stock TRL.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Switch back to training mode after inference sampling
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset= dataset,
    dataset_text_field = "text",         # Column containing formatted prompts
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 4,
    packing            = False,          # No sequence packing — cleaner training
    args = TrainingArguments(
        # ── Core hyperparameters (documented in report Table)
        per_device_train_batch_size   = BATCH_SIZE,
        gradient_accumulation_steps   = GRAD_ACCUM,  # Effective batch = 2×4 = 8
        max_steps                     = MAX_STEPS,
        learning_rate                 = LEARNING_RATE,
        optim                         = OPTIMIZER,   # adamw_8bit: memory-efficient
        lr_scheduler_type             = LR_SCHEDULER,
        warmup_steps                  = WARMUP_STEPS,

        # ── Precision — use bf16 if supported (Ampere+), else fp16 (T4)
        fp16  = not is_bfloat16_supported(),
        bf16  = is_bfloat16_supported(),

        # ── Logging & checkpointing
        logging_steps    = 25,           # Log loss every 25 steps
        save_steps       = 100,          # Save checkpoint every 100 steps
        output_dir       = f"/content/{MODEL_NAME}-checkpoints",
        save_total_limit = 2,            # Keep only 2 checkpoints (saves disk)

        # ── Reproducibility
        seed             = SEED,
        report_to        = "none",       # Disable W&B / TensorBoard in Colab
    ),
)

print("✅ Trainer configured.")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Max steps:            {MAX_STEPS}")
print(f"   Estimated duration:   ~15-25 min on T4")

In [ ]:
import time

print("🚀 Starting fine-tuning...")
print("-" * 60)

start_time = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start_time

print("-" * 60)
print(f"✅ Training complete in {elapsed/60:.1f} minutes.")
print(f"   Final training loss:  {trainer_stats.training_loss:.4f}")
print(f"   Total steps:          {trainer_stats.global_step}")

In [ ]:
import matplotlib.pyplot as plt
import json

# Extract loss values logged during training
log_history = trainer.state.log_history
steps  = [x["step"]         for x in log_history if "loss" in x]
losses = [x["loss"]         for x in log_history if "loss" in x]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, color="#2C7BB6", linewidth=2, marker="o", markersize=3)
plt.fill_between(steps, losses, alpha=0.15, color="#2C7BB6")
plt.title("Training Loss — Horus-OSINT (Llama-3-8B QLoRA)", fontsize=14, fontweight="bold")
plt.xlabel("Training Step")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/training_loss.png", dpi=150)
plt.show()
print("✅ Training loss curve saved to /content/training_loss.png")

## Section 9 — Capture Fine-Tuned Model Response (After Training)
We run the **exact same query** used in Section 7 to demonstrate domain specialization.

In [ ]:
# Enable fast inference mode on the now-trained model
FastLanguageModel.for_inference(model)

finetuned_response = generate_response(TEST_PROMPT)

print("=" * 60)
print("🦅 HORUS-OSINT FINE-TUNED RESPONSE (after training):")
print("=" * 60)
print(finetuned_response)
print("=" * 60)

In [ ]:
# Side-by-side qualitative comparison for the report
print("\n" + "=" * 60)
print("  QUALITATIVE COMPARISON — Base Model vs Horus-OSINT")
print("=" * 60)
print(f"\n📌 QUERY:\n{TEST_PROMPT}")
print("\n" + "-" * 60)
print("❌ BASE MODEL (generic, unstructured):")
print("-" * 60)
print(base_response)
print("\n" + "-" * 60)
print("✅ HORUS-OSINT (structured intelligence report):")
print("-" * 60)
print(finetuned_response)
print("\n" + "=" * 60)
print("Screenshot this cell output for Section 5 of the report!")

## Section 10 — Export to GGUF Format
Unsloth's native exporter converts the fine-tuned LoRA weights directly to GGUF
(4-bit quantized) — the format required by Ollama on EC2. This avoids using `llama.cpp` manually.

In [ ]:
import os

GGUF_PATH = f"/content/{MODEL_NAME}"

print(f"⚙️  Exporting to GGUF (q4_k_m quantization)...")
print("    This may take 5-10 minutes.")

# Unsloth's native GGUF export — merges LoRA into base, then quantizes
# q4_k_m = 4-bit K-quant medium — best quality/size trade-off for Ollama
model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method = "q4_k_m",
)

# List exported files
gguf_file = f"{GGUF_PATH}/horus-llama3-osint-Q4_K_M.gguf"
if not os.path.exists(gguf_file):
    # Unsloth sometimes uses a slightly different filename pattern
    import glob
    matches = glob.glob(f"{GGUF_PATH}/*.gguf")
    gguf_file = matches[0] if matches else gguf_file

size_gb = os.path.getsize(gguf_file) / 1e9
print(f"\n✅ GGUF export complete.")
print(f"   File: {gguf_file}")
print(f"   Size: {size_gb:.2f} GB")

## Section 11 — Upload GGUF to S3
The GGUF file is uploaded to S3 so the EC2 deployment step (Section 6) can
download it without Colab staying connected.

In [ ]:
import boto3
from botocore.exceptions import ClientError

# Re-init S3 client (credentials already set in os.environ)
s3 = boto3.client("s3", region_name=AWS_REGION)

S3_GGUF_KEY = f"models/{MODEL_NAME}-Q4_K_M.gguf"

print(f"⬆️  Uploading GGUF to s3://{S3_BUCKET}/{S3_GGUF_KEY} ...")
print(f"    File size: {size_gb:.2f} GB — this may take a few minutes.")

# Upload with progress callback
uploaded_bytes = {"n": 0}
total_bytes    = os.path.getsize(gguf_file)

def progress(chunk):
    uploaded_bytes["n"] += chunk
    pct = 100 * uploaded_bytes["n"] / total_bytes
    print(f"   {pct:.1f}%", end="\r")

s3.upload_file(
    gguf_file,
    S3_BUCKET,
    S3_GGUF_KEY,
    Callback=progress,
)

print(f"\n✅ Upload complete!")
print(f"   S3 URI: s3://{S3_BUCKET}/{S3_GGUF_KEY}")
print()
print("Next step — EC2 download command:")
print(f'   aws s3 cp s3://{S3_BUCKET}/{S3_GGUF_KEY} ./{MODEL_NAME}.gguf')

## Section 12 — Hyperparameter Summary Table
Required for the project report (Section 5).

In [ ]:
import pandas as pd

hp_table = pd.DataFrame([
    {"Hyperparameter":"Learning Rate",           "Value":"2e-4",      "Reasoning":"Standard, stable starting point for PEFT using AdamW"},
    {"Hyperparameter":"Batch Size",               "Value":"2",         "Reasoning":"Optimised for memory efficiency on Colab T4 16GB VRAM"},
    {"Hyperparameter":"Gradient Accumulation",    "Value":"4",         "Reasoning":"Simulates effective batch size of 8 for stable gradient updates"},
    {"Hyperparameter":"Effective Batch Size",     "Value":"8",         "Reasoning":"2 × 4 = 8; provides stable gradient estimates"},
    {"Hyperparameter":"LoRA Rank (r)",            "Value":"16",        "Reasoning":"Balances VRAM usage with expressive power needed for domain adaptation"},
    {"Hyperparameter":"LoRA Alpha",               "Value":"32",        "Reasoning":"Standard scaling factor = 2 × rank applied to LoRA weights"},
    {"Hyperparameter":"LoRA Dropout",             "Value":"0.05",      "Reasoning":"Light regularisation to prevent overfitting on OSINT format"},
    {"Hyperparameter":"Optimizer",                "Value":"adamw_8bit","Reasoning":"Highly memory-efficient optimizer provided natively by Unsloth"},
    {"Hyperparameter":"Max Steps",                "Value":"500",       "Reasoning":"Sufficient for the model to adapt to the OSINT instruction format"},
    {"Hyperparameter":"Warmup Steps",             "Value":"50",        "Reasoning":"Gradual LR ramp-up (10% of training) to prevent early instability"},
    {"Hyperparameter":"LR Scheduler",             "Value":"cosine",    "Reasoning":"Smooth decay avoids sharp LR drops that destabilise PEFT training"},
    {"Hyperparameter":"Max Sequence Length",      "Value":"2048",      "Reasoning":"Covers longest OSINT Q&A pairs with comfortable margin"},
    {"Hyperparameter":"Quantization (training)",  "Value":"NF4 (4-bit)","Reasoning":"QLoRA base quantization — reduces VRAM from ~16GB to ~6GB"},
    {"Hyperparameter":"Quantization (export)",    "Value":"q4_k_m",    "Reasoning":"Best quality/size tradeoff for Ollama deployment on g4dn.xlarge"},
])

pd.set_option("display.max_colwidth", 80)
print("=" * 60)
print("  HYPERPARAMETER TABLE — Horus-OSINT Fine-Tuning")
print("=" * 60)
print(hp_table.to_string(index=False))

## ✅ Notebook Complete

### What was produced:
| Output | Location |
|--------|----------|
| Fine-tuned GGUF model | `s3://25BBDF-bucket/models/horus-llama3-osint-Q4_K_M.gguf` |
| Training loss curve | `/content/training_loss.png` |
| Base model response | Captured in Section 7 |
| Fine-tuned response | Captured in Section 9 |

### Next step — EC2 Deployment (Section 6 of report):
```bash
# On EC2 instance:
aws s3 cp s3://25BBDF-bucket/models/horus-llama3-osint-Q4_K_M.gguf ./horus-llama3-osint.gguf
echo "FROM ./horus-llama3-osint.gguf" > Modelfile
ollama create horus-osint -f Modelfile
ollama run horus-osint
```